# 23 — Mini Project: E-Commerce System Architecture

## Objectives
- Apply all OOP concepts in a real project
- Design a complete system from requirements to code
- Practice LLD with multiple interacting classes

## System Requirements
Build a simplified e-commerce order system:
- **Product**: catalog, inventory
- **Customer**: profile, address, payment methods
- **Order**: items, total, status lifecycle
- **Payment**: multiple methods (UPI, Card)
- **Notification**: email/SMS on events

In [3]:
import java.util.*;
import java.time.LocalDateTime;

// Domain entities
record Product(String id, String name, double price, int stock) {
    boolean isAvailable(int qty) { return stock >= qty; }
}

record Address(String street, String city, String pincode, String state) {
    public String toString() { return street + ", " + city + " - " + pincode + ", " + state; }
}

class Customer {
    private final String customerId;
    private final String name;
    private final String email;
    private Address defaultAddress;
    
    Customer(String id, String name, String email) {
        this.customerId = id; this.name = name; this.email = email;
    }
    
    void setAddress(Address addr) { this.defaultAddress = addr; }
    String getName() { return name; }
    String getEmail() { return email; }
    Address getAddress() { return defaultAddress; }
    String getCustomerId() { return customerId; }
}

enum OrderStatus { CREATED, CONFIRMED, SHIPPED, DELIVERED, CANCELLED }

class OrderItem {
    Product product; int quantity;
    OrderItem(Product p, int qty) { product = p; quantity = qty; }
    double subtotal() { return product.price() * quantity; }
    public String toString() { return product.name() + " x" + quantity + " = INR " + subtotal(); }
}

class Order {
    private static int counter = 1000;
    private final String orderId;
    private final Customer customer;
    private final List<OrderItem> items;
    private OrderStatus status;
    private final LocalDateTime createdAt;
    
    Order(Customer customer) {
        this.orderId = "ORD-" + (++counter);
        this.customer = customer;
        this.items = new ArrayList<>();
        this.status = OrderStatus.CREATED;
        this.createdAt = LocalDateTime.now();
    }
    
    void addItem(Product p, int qty) {
        if (!p.isAvailable(qty)) { System.out.println("Insufficient stock for: " + p.name()); return; }
        items.add(new OrderItem(p, qty));
    }
    
    double getTotal() { return items.stream().mapToDouble(OrderItem::subtotal).sum(); }
    
    void updateStatus(OrderStatus newStatus) {
        System.out.printf("[Order %s] Status: %s → %s%n", orderId, status, newStatus);
        this.status = newStatus;
    }
    
    void printSummary() {
        System.out.println("\n=== Order Summary: " + orderId + " ===");
        System.out.println("Customer : " + customer.getName() + " (" + customer.getEmail() + ")");
        System.out.println("Deliver  : " + customer.getAddress());
        System.out.println("--- Items ---");
        items.forEach(i -> System.out.println("  " + i));
        System.out.printf("TOTAL    : INR %.2f%n", getTotal());
        System.out.println("Status   : " + status);
    }
}

public class Main {
    public static void main(String[] args) {
        // Demo - Note the escaped string: "Laptop 15\" i7"
        Product laptop = new Product("P001", "Laptop 15\" i7", 65000.0, 10);
        Product mouse  = new Product("P002", "Wireless Mouse", 799.0, 50);
        Product bag    = new Product("P003", "Laptop Bag", 1299.0, 30);

        Customer customer = new Customer("C001", "Arjun Sharma", "arjun@gmail.com");
        customer.setAddress(new Address("42 MG Road", "Bangalore", "560001", "Karnataka"));

        Order order = new Order(customer);
        order.addItem(laptop, 1);
        order.addItem(mouse, 2);
        order.addItem(bag, 1);
        order.printSummary();

        order.updateStatus(OrderStatus.CONFIRMED);
        order.updateStatus(OrderStatus.SHIPPED);
        order.updateStatus(OrderStatus.DELIVERED);
    }
}

## Next Steps
- Add `PaymentService` using the `Payment` interface from Notebook 07
- Add `NotificationService` using Observer pattern
- Add persistence layer (Repository pattern)
- Write JUnit tests for all classes